# 🛡️ Glu-Stock: 00b_MODEL_RETRAINING_CNN
**Phase**: Dynamic Cross-Sectional Intelligence (CNN Brain)

This notebook trains the CNN 'Super Brain' on a dynamic panel dataset containing 5 years of historical data from the latest active LQ45 constituents. It scrapes the current LQ45 members to ensure the deep learning model spots structural patterns on highly liquid assets.

In [ ]:
!pip install -q yfinance firebase-admin pandas tensorflow python-dotenv

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Web Fetchers)
import json, os, firebase_admin, joblib, pandas as pd, yfinance as yf
import numpy as np
import tensorflow as tf
from firebase_admin import credentials, firestore
from datetime import datetime

    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try: tg = user_secrets.get_secret("TELEGRAM_TOKEN")
            except: tg = None
            return {
                "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
                "telegram": tg
            }
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "telegram": os.getenv("TELEGRAM_TOKEN")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
        
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def get_history(self, limit=5):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        history = [doc.to_dict() for doc in docs]
        return {str(i): h for i, h in enumerate(reversed(history))} if history else {}
        
    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [doc.to_dict() for doc in docs]
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/lq45.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as url:
            data = json.loads(url.read().decode())
            return [f"{t}.JK" for t in data]
    except Exception as e:
        print(f"⚠️ Github LQ45 fetch failed: {e}. Using highly-curated fallback LQ45 list.")
    return fallback

def get_full_idx_universe():
    print("🌐 Fetching ALL IDX listed companies from Official IDX API...")
    fallback = ['AALI.JK', 'ABMM.JK', 'ACES.JK', 'ADHI.JK', 'AISA.JK', 'AKRA.JK', 'AMRT.JK', 'ANTM.JK', 'APLN.JK', 'ARNA.JK', 'ARTO.JK', 'ASGR.JK', 'ASII.JK', 'ASRI.JK', 'ASSA.JK', 'AUTO.JK', 'BACA.JK', 'BALI.JK', 'BAYU.JK', 'BBCA.JK', 'BBHI.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BBYB.JK', 'BCAP.JK', 'BDMN.JK', 'BEST.JK', 'BFIN.JK', 'BGTG.JK', 'BINA.JK', 'BIRD.JK', 'BISI.JK', 'BJBR.JK', 'BJTM.JK', 'BKSL.JK', 'BMRI.JK', 'BMTR.JK', 'BNGA.JK', 'BNII.JK', 'BNLI.JK', 'BRMS.JK', 'BRPT.JK', 'BSDE.JK', 'BSIM.JK', 'BTPN.JK', 'BUDI.JK', 'BUKK.JK', 'BUMI.JK', 'BVIC.JK', 'BWPT.JK', 'BYAN.JK', 'CASS.JK', 'CFIN.JK', 'CITA.JK', 'CMNP.JK', 'CPIN.JK', 'CTRA.JK', 'DEWA.JK', 'DILD.JK', 'DLTA.JK', 'DMAS.JK', 'DNET.JK', 'DOID.JK', 'DSNG.JK', 'DSSA.JK', 'ELSA.JK', 'EMTK.JK', 'ENRG.JK', 'ERAA.JK', 'ESSA.JK', 'EXCL.JK', 'GEMS.JK', 'GGRM.JK', 'GJTL.JK', 'GWSA.JK', 'HEXA.JK', 'HMSP.JK', 'HRUM.JK', 'ICBP.JK', 'IMAS.JK', 'IMPC.JK', 'INCO.JK', 'INDF.JK', 'INDY.JK', 'INKP.JK', 'INPC.JK', 'INTP.JK', 'ISAT.JK', 'ISSP.JK', 'ITMG.JK', 'JKON.JK', 'JPFA.JK', 'JRPT.JK', 'JSMR.JK', 'JTPE.JK', 'KBLI.JK', 'KIJA.JK', 'KKGI.JK', 'KLBF.JK', 'KPIG.JK', 'LPKR.JK', 'LPPF.JK', 'LSIP.JK', 'LTLS.JK', 'MAIN.JK', 'MAPI.JK', 'MAYA.JK', 'MBSS.JK', 'MCOR.JK', 'MDKA.JK', 'MEDC.JK', 'MEGA.JK', 'MIDI.JK', 'MIKA.JK', 'MLBI.JK', 'MLIA.JK', 'MLPL.JK', 'MMLP.JK', 'MNCN.JK', 'MPMX.JK', 'MREI.JK', 'MTDL.JK', 'MTLA.JK', 'MYOR.JK', 'NISP.JK', 'PANR.JK', 'PANS.JK', 'PGAS.JK', 'PNBN.JK', 'PNIN.JK', 'PNLF.JK', 'PTBA.JK', 'PTPP.JK', 'PTRO.JK', 'PWON.JK', 'RAJA.JK', 'RALS.JK', 'SAME.JK', 'SCMA.JK', 'SGRO.JK', 'SIDO.JK', 'SILO.JK', 'SIMP.JK', 'SMAR.JK', 'SMBR.JK', 'SMDR.JK', 'SMGR.JK', 'SMMA.JK', 'SMRA.JK', 'SMSM.JK', 'SRTG.JK', 'SSIA.JK', 'SSMS.JK', 'TBIG.JK', 'TBLA.JK', 'TINS.JK', 'TKIM.JK', 'TLKM.JK', 'TMAS.JK', 'TOBA.JK', 'TOTL.JK', 'TOWR.JK', 'TPMA.JK', 'TRIM.JK', 'TSPC.JK', 'ULTJ.JK', 'UNIC.JK', 'UNTR.JK', 'UNVR.JK', 'VICO.JK', 'WIIM.JK', 'WINS.JK', 'WTON.JK', 'SHIP.JK', 'POWR.JK', 'PRDA.JK', 'BRIS.JK', 'CARS.JK', 'CLEO.JK', 'WOOD.JK', 'HRTA.JK', 'MARK.JK', 'MCAS.JK', 'PSSI.JK', 'MORA.JK', 'PBID.JK', 'IPCM.JK', 'BTPS.JK', 'SPTO.JK', 'HEAL.JK', 'TUGU.JK', 'MSIN.JK', 'MAPA.JK', 'IPCC.JK', 'FILM.JK', 'PANI.JK', 'GOOD.JK', 'SKRN.JK', 'BOLA.JK', 'KOTA.JK', 'KEEN.JK', 'TEBE.JK', 'KEJU.JK', 'PSGO.JK', 'UCID.JK', 'CSRA.JK', 'SAMF.JK', 'SGER.JK', 'PNGO.JK', 'BBSI.JK', 'VICI.JK', 'WMUU.JK', 'UNIQ.JK', 'TAPG.JK', 'BMHS.JK', 'MCOL.JK', 'GTSI.JK', 'MTEL.JK', 'CMRY.JK', 'RMKE.JK', 'AVIA.JK', 'DRMA.JK', 'ADMR.JK', 'STAA.JK', 'MTMH.JK', 'TRGU.JK', 'HATM.JK', 'JARR.JK', 'ELPI.JK', 'MKTR.JK', 'OMED.JK', 'SUNI.JK', 'PGEO.JK', 'BDKR.JK', 'CUAN.JK', 'SMIL.JK', 'AMMN.JK', 'MAHA.JK', 'ERAL.JK', 'BREN.JK', 'MSTI.JK', 'ALII.JK', 'GOLF.JK', 'DAAZ.JK', 'AADI.JK', 'MDIY.JK', 'DGWG.JK', 'CBDK.JK', 'MINE.JK', 'PSAT.JK', 'BLOG.JK', 'YUPI.JK', 'MDLA.JK', 'NCKL.JK', 'MBMA.JK', 'RAAM.JK', 'ADRO.JK', 'AGRO.JK']
    
    # 1. Try Official IDX API
    try:
        import urllib.request
        hdrs = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Referer': 'https://www.idx.co.id/'
        }
        req = urllib.request.Request('https://www.idx.co.id/primary/StockData/GetSecuritiesStock?length=9999', headers=hdrs)
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            if 'data' in data:
                tickers = [f"{t['Code']}.JK" for t in data['data'] if 'Code' in t]
                if tickers:
                    print(f"✅ Successfully fetched {len(tickers)} companies from IDX Official API.")
                    return list(set(tickers)) 
    except Exception as e:
        print(f"⚠️ Official IDX API failed: {e}. Trying Github Proxy...")
        
    # 2. Try Github Alternative
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/stock-list.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            tickers = [f"{t['ticker']}.JK" for t in data if 'ticker' in t]
            if tickers:
                print(f"✅ Successfully fetched {len(tickers)} companies from Github proxy.")
                return list(set(tickers))
    except Exception as e:
        print(f"⚠️ Full fetch failed: {e}. Falling back to MASTER 259 Papan Utama list.")
        
    return fallback


In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Lightweight CNN Pipeline)
import warnings
warnings.filterwarnings('ignore')

def build_cnn_panel_data(universe, seq_len=30, period='5y'):
    all_X, all_y = [], []
    n_total = len(universe)
    print(f'\U0001f4c9 Fetching {period} of data for {n_total} tickers...')
    skip_empty, skip_short, fail_err, ok_count = 0, 0, 0, 0
    for idx, ticker in enumerate(universe):
        if idx % 50 == 0:
            print(f'  ... processing {idx}/{n_total} ...')
        try:
            df = yf.download(ticker, period=period, progress=False, auto_adjust=True)
            if df is None or len(df) == 0:
                skip_empty += 1
                continue
            if len(df) < seq_len + 10:
                skip_short += 1
                continue
            close = df['Close'].squeeze()
            high = df['High'].squeeze()
            low = df['Low'].squeeze()
            volume = df['Volume'].squeeze()
            opn = df['Open'].squeeze()
            if hasattr(close, 'ndim') and close.ndim > 1:
                close = close.iloc[:, 0]
            if hasattr(high, 'ndim') and high.ndim > 1:
                high = high.iloc[:, 0]
            if hasattr(low, 'ndim') and low.ndim > 1:
                low = low.iloc[:, 0]
            if hasattr(volume, 'ndim') and volume.ndim > 1:
                volume = volume.iloc[:, 0]
            if hasattr(opn, 'ndim') and opn.ndim > 1:
                opn = opn.iloc[:, 0]
            raw = np.column_stack([opn.values, high.values, low.values, close.values, volume.values])
            close_vals = close.values
            ticker_x, ticker_y = [], []
            for i in range(len(raw) - seq_len - 1):
                seq = raw[i:i+seq_len]
                seq_min = seq.min(axis=0)
                seq_max = seq.max(axis=0)
                norm_seq = (seq - seq_min) / (seq_max - seq_min + 1e-7)
                ticker_x.append(norm_seq)
                y_val = 1 if close_vals[i+seq_len+1] > close_vals[i+seq_len] else 0
                ticker_y.append(y_val)
            if len(ticker_x) > 0:
                all_X.extend(ticker_x)
                all_y.extend(ticker_y)
                ok_count += 1
        except Exception as e:
            fail_err += 1
            if fail_err <= 5:
                print(f'\u26a0\ufe0f {ticker}: {type(e).__name__}: {e}')
            continue
    print(f'\U0001f4ca Pipeline: OK={ok_count} | Empty={skip_empty} | Short={skip_short} | Error={fail_err}')
    if len(all_X) == 0:
        print(f'\u274c No valid data!')
        return None, None
    print(f'\u2705 Aggregated {ok_count}/{n_total} tickers | {len(all_y)} sequences.')
    return np.array(all_X), np.array(all_y)

def train_cnn(X, y):
    print(f'\U0001f9e0 Training Lightweight CNN on {len(y)} sequences...')
    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same', input_shape=(30, 5)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    split_idx = int(len(X) * 0.85)
    X_tr, X_te = X[:split_idx], X[split_idx:]
    y_tr, y_te = y[:split_idx], y[split_idx:]
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
    history = model.fit(X_tr, y_tr, epochs=20, batch_size=256, validation_data=(X_te, y_te), callbacks=[early_stop], verbose=1)
    t_acc = max(history.history['accuracy'])
    v_acc = max(history.history['val_accuracy'])
    return model, t_acc, v_acc


In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_retrain():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    output_dir = '/kaggle/working/'
    
    universe = get_full_idx_universe()
    
    X_all, y_all = build_cnn_panel_data(universe)
    if X_all is None:
        print('\u26a0\ufe0f Retraining aborted.')
        return
    
    cnn_model, t_acc, v_acc = train_cnn(X_all, y_all)
    
    print('\u2699\ufe0f Converting to TFLite...')
    converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
    tflite_model = converter.convert()
    with open(os.path.join(output_dir, 'cnn_daily_t2.tflite'), 'wb') as f:
        f.write(tflite_model)
    
    log_lines = [
        'CNN Training Complete.',
        f'\U0001f393 Training Acc: {t_acc:.2%}',
        f'\U0001f6e1\ufe0f OOS Validation Acc: {v_acc:.2%}',
        f'\U0001f4ca Universe: {len(universe)} symbols | Sequences: {len(y_all)}',
        f'\U0001f3af Input: {X_all.shape} | Output: cnn_daily_t2.tflite ({len(tflite_model)} bytes)',
    ]
    fb.log_event('RETRAINING_CNN', chr(10).join(log_lines))
    for line in log_lines:
        print(line)

run_retrain()
